# 04 · 資料增強:對抗過擬合

模型在訓練集上很準、在測試集上卻爛掉——這就是**過擬合**:它把訓練圖「背」起來了,而不是學到通則。

**資料增強(data augmentation)** 是視覺領域對抗過擬合最有效的武器:

> 每次訓練時,把影像**隨機**裁切、翻轉、調色一下。模型每個 epoch 看到的都是「同一張圖的不同變體」,等於免費擴增了資料,被迫學到**真正穩健**的特徵,而不是背圖。

## 1. 安裝與資料

In [ ]:
%pip install -q -U torchvision

In [ ]:
# CIFAR-10 的 10 個類別，以及這個資料集慣用的正規化平均/標準差。
CIFAR10_CLASSES = ["plane", "car", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

In [ ]:
import matplotlib.pyplot as plt
import torch


def show_images(imgs, labels=None, classes=None, cols=8, denorm=None):
    """把一批張量影像 [N,C,H,W] 畫成 grid。

    denorm=(mean,std) 時會先反正規化回 0~1 再顯示（不然正規化過的圖會怪怪的）。
    """
    imgs = imgs.detach().cpu()
    n = len(imgs)
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols * 1.3, rows * 1.45))
    for i in range(n):
        img = imgs[i]
        if denorm is not None:
            mean, std = denorm
            img = img * torch.tensor(std).view(-1, 1, 1) + torch.tensor(mean).view(-1, 1, 1)
        img = img.clamp(0, 1).permute(1, 2, 0).numpy()
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(img)
        ax.axis("off")
        if labels is not None:
            lab = labels[i].item() if hasattr(labels[i], "item") else labels[i]
            ax.set_title(classes[lab] if classes else str(lab), fontsize=8)
    plt.tight_layout()
    plt.show()

## 2. 定義增強管線

常見的視覺增強:隨機裁切(補邊後裁)、隨機水平翻轉、顏色抖動。**注意:增強只用在訓練集,測試集要保持原樣**(評估要公平)。

In [ ]:
from torchvision import transforms

train_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),       # 補 4 像素邊再隨機裁回 32
    transforms.RandomHorizontalFlip(),          # 隨機左右翻
    transforms.ColorJitter(0.2, 0.2, 0.2),      # 隨機調亮度/對比/飽和
    transforms.ToTensor(),
])
test_plain = transforms.Compose([transforms.ToTensor()])

## 3. 親眼看增強:同一張圖的 8 個隨機變體

每次取出都不一樣——這正是模型每個 epoch 看到的。

In [ ]:
import torch
import torchvision

raw = torchvision.datasets.CIFAR10("./data", train=True, download=True)
pil_img, label = raw[7]                          # 取一張原圖(PIL)
variants = torch.stack([train_aug(pil_img) for _ in range(8)])
print("原圖類別:", CIFAR10_CLASSES[label])
show_images(variants, cols=8)

## 4. 過擬合的樣子:訓練 vs 驗證的差距

沒有增強時,模型很容易訓練準確率衝很高、但測試準確率卡住——兩者拉開的**那道縫**就是過擬合。把上面的 `train_aug` 換進 DataLoader 訓練,通常能把這道縫縮小、測試準確率往上。

In [ ]:
# 觀念示意:訓練時用 train_aug(資料增強)、評估時用 test_plain(原樣)。
# train_set = torchvision.datasets.CIFAR10("./data", train=True, transform=train_aug)
# test_set  = torchvision.datasets.CIFAR10("./data", train=False, transform=test_plain)
print("規則:增強只加在訓練集；測試集保持原樣,評估才公平。")
print("加了增強後,train/val 準確率的差距(過擬合的縫)會縮小。")

## 小結

- **過擬合 = 背訓練圖、學不到通則**,徵兆是訓練準確率 ≫ 測試準確率。
- **資料增強**用隨機裁切/翻轉/調色,讓模型每次看到不同變體,被迫學穩健特徵。
- 鐵則:**增強只用於訓練集**,測試集保持原樣。
- 搭配上一課的遷移學習,是榨出視覺模型效能的標準組合。

下一課:跳出「分類」——讓模型不只說「是什麼」,還要框出「在哪裡」:**物件偵測**。